# TaxGPT — Episode 9: Decoder-Only Architecture

Companion notebook to blog post *"Decoder-Only Transformers Explained: Why TaxGPT Skips the Encoder (Episode 9)"*.

Builds a minimal encoder (bidirectional self-attention) and a minimal decoder (causal self-attention + cross-attention) side by side, so the actual mechanical difference between "decoder-only" and "encoder-decoder" is visible in code rather than just described.

Reference: Vaswani et al., *Attention Is All You Need* (2017); Sebastian Raschka, *Build a Large Language Model (From Scratch)*, Ch. 1 & 4.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(0)

## 1. Bidirectional self-attention (what an ENCODER uses)

No causal mask — every position can attend to every other position, both directions. This is what "reads the whole input at once" means mechanically.

In [2]:
class BidirectionalSelfAttention(nn.Module):
    """No mask -- every token attends to every other token, both directions."""
    def __init__(self, emb_dim, head_dim):
        super().__init__()
        self.W_q = nn.Linear(emb_dim, head_dim, bias=False)
        self.W_k = nn.Linear(emb_dim, head_dim, bias=False)
        self.W_v = nn.Linear(emb_dim, head_dim, bias=False)
        self.head_dim = head_dim

    def forward(self, x):
        Q, K, V = self.W_q(x), self.W_k(x), self.W_v(x)
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.head_dim)
        attn_weights = F.softmax(scores, dim=-1)   # no masking at all
        return attn_weights @ V, attn_weights

enc_attn = BidirectionalSelfAttention(emb_dim=32, head_dim=32)
x = torch.randn(1, 5, 32)
_, enc_weights = enc_attn(x)

print("encoder attention matrix (rows=query pos, cols=key pos):")
print(enc_weights[0].round(decimals=3))
print()
print("upper triangle (attention to FUTURE positions) sum:", torch.triu(enc_weights[0], diagonal=1).sum().item())
print("-> nonzero: encoder tokens freely attend forward AND backward")

encoder attention matrix (rows=query pos, cols=key pos):
tensor([[0.1690, 0.1470, 0.2770, 0.2210, 0.1860],
        [0.2420, 0.1960, 0.1140, 0.1960, 0.2510],
        [0.1690, 0.1870, 0.1740, 0.2090, 0.2600],
        [0.2600, 0.1310, 0.1750, 0.2090, 0.2250],
        [0.2690, 0.2220, 0.1880, 0.1910, 0.1300]], grad_fn=<RoundBackward1>)

upper triangle (attention to FUTURE positions) sum: 2.08701229095459
-> nonzero: encoder tokens freely attend forward AND backward


## 2. Causal self-attention (what a DECODER uses) — recap from Episode 4

In [3]:
class CausalSelfAttention(nn.Module):
    def __init__(self, emb_dim, head_dim, context_len):
        super().__init__()
        self.W_q = nn.Linear(emb_dim, head_dim, bias=False)
        self.W_k = nn.Linear(emb_dim, head_dim, bias=False)
        self.W_v = nn.Linear(emb_dim, head_dim, bias=False)
        self.head_dim = head_dim
        self.register_buffer("mask", torch.tril(torch.ones(context_len, context_len)))

    def forward(self, x):
        B, T, C = x.shape
        Q, K, V = self.W_q(x), self.W_k(x), self.W_v(x)
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.head_dim)
        scores = scores.masked_fill(self.mask[:T, :T] == 0, float('-inf'))
        attn_weights = F.softmax(scores, dim=-1)
        return attn_weights @ V, attn_weights

dec_self_attn = CausalSelfAttention(emb_dim=32, head_dim=32, context_len=128)
_, dec_weights = dec_self_attn(x)

print("decoder causal attention matrix:")
print(dec_weights[0].round(decimals=3))
print()
print("upper triangle (attention to FUTURE positions) sum:", torch.triu(dec_weights[0], diagonal=1).sum().item())
print("-> exactly 0: decoder tokens can never see the future")

decoder causal attention matrix:
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4820, 0.5180, 0.0000, 0.0000, 0.0000],
        [0.3840, 0.2180, 0.3980, 0.0000, 0.0000],
        [0.1900, 0.3560, 0.2680, 0.1860, 0.0000],
        [0.1350, 0.3010, 0.2000, 0.1880, 0.1760]], grad_fn=<RoundBackward1>)

upper triangle (attention to FUTURE positions) sum: 0.0
-> exactly 0: decoder tokens can never see the future


## 3. Cross-attention: the mechanism that ONLY exists in encoder-decoder models

Cross-attention looks like self-attention, except Query comes from the decoder's own sequence while Key and Value come from the encoder's finished output. This is the bridge between the two stacks -- and it's exactly what a decoder-only model has no use for, since there's no second sequence to bridge to.

In [4]:
class CrossAttention(nn.Module):
    """Query from the decoder sequence; Key/Value from the encoder's output."""
    def __init__(self, emb_dim, head_dim):
        super().__init__()
        self.W_q = nn.Linear(emb_dim, head_dim, bias=False)
        self.W_k = nn.Linear(emb_dim, head_dim, bias=False)
        self.W_v = nn.Linear(emb_dim, head_dim, bias=False)
        self.head_dim = head_dim

    def forward(self, decoder_x, encoder_out):
        Q = self.W_q(decoder_x)      # from the decoder's own sequence
        K = self.W_k(encoder_out)    # from the ENCODER's finished output
        V = self.W_v(encoder_out)
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.head_dim)
        attn_weights = F.softmax(scores, dim=-1)   # causal masking isn't meaningful here --
        return attn_weights @ V, attn_weights        # the encoder output is fully available already

# toy source sequence (e.g. "English sentence", 7 tokens) and target sequence (5 tokens)
source_x = torch.randn(1, 7, 32)
target_x = torch.randn(1, 5, 32)

encoder_out, _ = enc_attn(source_x)               # encoder reads the full source, bidirectionally
cross_attn = CrossAttention(emb_dim=32, head_dim=32)
cross_out, cross_weights = cross_attn(target_x, encoder_out)

print("decoder sequence length:", target_x.shape[1])
print("encoder sequence length:", source_x.shape[1])
print("cross-attention weights shape:", cross_weights.shape, " -> (decoder_len, encoder_len), NOT square")
print("cross-attention output shape:", cross_out.shape, " -> matches decoder sequence length")

decoder sequence length: 5
encoder sequence length: 7
cross-attention weights shape: torch.Size([1, 5, 7])  -> (decoder_len, encoder_len), NOT square
cross-attention output shape: torch.Size([1, 5, 32])  -> matches decoder sequence length


Notice the cross-attention weight matrix is `(5, 7)` -- not square like self-attention's `(T, T)`. That asymmetry is the whole point: it's explicitly relating two *different* sequences (decoder positions as queries, encoder positions as keys/values), which only makes sense when there are genuinely two sequences to relate. TaxGPT has one continuous stream of GST text -- there's no second sequence for this mechanism to attach to.

## 4. Side by side: what each architecture actually computes

In [5]:
print("ENCODER block (bidirectional):")
print("  self-attention: unmasked -- every position sees every other position")
print("  used for: reading a fixed input in full, before generation starts")
print()
print("DECODER block, encoder-decoder style:")
print("  self-attention: causal -- only sees itself so far")
print("  + cross-attention: causal decoder queries attend to the FULL encoder output")
print("  used for: generating a target sequence conditioned on a separate source")
print()
print("DECODER-ONLY block (TaxGPT, Episode 8's TransformerBlock):")
print("  self-attention: causal -- only sees itself so far")
print("  NO cross-attention -- there's no second sequence to attend to")
print("  used for: continuing one single stream of text, next-token prediction")

ENCODER block (bidirectional):
  self-attention: unmasked -- every position sees every other position
  used for: reading a fixed input in full, before generation starts

DECODER block, encoder-decoder style:
  self-attention: causal -- only sees itself so far
  + cross-attention: causal decoder queries attend to the FULL encoder output
  used for: generating a target sequence conditioned on a separate source

DECODER-ONLY block (TaxGPT, Episode 8's TransformerBlock):
  self-attention: causal -- only sees itself so far
  NO cross-attention -- there's no second sequence to attend to
  used for: continuing one single stream of text, next-token prediction


## Takeaway

The mechanical difference between decoder-only and encoder-decoder isn't philosophical -- it's exactly these two things: whether self-attention is masked (causal) or not (bidirectional), and whether a cross-attention mechanism exists to bridge two separate sequences. TaxGPT needs the first (causal, so it can generate left to right) and has no use for the second (no separate source sequence exists in a next-token-prediction task over one continuous corpus).

**Next notebook: Episode 10 — Assembling the full 131M-parameter model.**